In [1]:
import os
import sys
import json
project_dir = os.path.dirname(os.getcwd())
sys.path.append(project_dir)

import numpy as np  # Ensure this is imported if not already


import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = torch.Generator().manual_seed(42)
print(device)

cuda


# Data

In [2]:
from data.cifar10 import get_cifar10_pipeline

train_loader, val_loader, test_loader = get_cifar10_pipeline(batch_size=128, indexed=True)
sample_x, sample_y, idx = next(iter(train_loader))
print(sample_x.shape)
print(sample_y.shape)
print(idx.shape)

Files already downloaded and verified
Files already downloaded and verified
torch.Size([128, 3, 32, 32])
torch.Size([128])
torch.Size([128])


# Self Distill

In [ ]:
import tqdm
from utils.train import evaluate_model
from utils.losses import DistillationLoss, Accuracy


def self_distill_model(train_loader, model, criterion, optimizer, scheduler, 
                       device, teacher_logits_dict, epoch=0, warmup_epochs=10):
    model.train()
    epoch_loss = []

    for inputs, labels, indices in tqdm.tqdm(train_loader, desc='training...', file=sys.stdout):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)

        # Determine if we can apply self-distillation
        if epoch >= warmup_epochs and all(idx.item() in teacher_logits_dict for idx in indices):
            prev_logits = torch.stack([teacher_logits_dict[idx.item()] for idx in indices]).to(device)
            loss = criterion(outputs, prev_logits.detach(), labels)
        else:
            loss = F.cross_entropy(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Store current logits for self-distillation in next epoch
        for i, idx in enumerate(indices):
            teacher_logits_dict[idx.item()] = outputs[i].detach().clone()

        epoch_loss.append(loss.item())

    scheduler.step()
    return np.mean(epoch_loss)


def distill_val(train_loader, val_loader, model, criterion, optimizer, scheduler, 
                device='cpu', aux_metrics={}, path="./temp.pth", patience=50, epochs=50):
    metrics = {"train_loss": [], "accuracy": []}
    for k in aux_metrics.keys():
        metrics[k] = []

    teacher_logits_dict = {}
    best_val_acc = 0
    counter = 0
    for epoch in range(epochs):
        train_loss = self_distill_model(train_loader, model, criterion, optimizer, scheduler, device, teacher_logits_dict, epoch=epoch)
        val_acc = evaluate_model(val_loader, model, Accuracy(), device)
        metrics['train_loss'].append(train_loss)
        metrics['accuracy'].append(val_acc)
        for k, v in aux_metrics.items():
            metrics[k].append(evaluate_model(val_loader, model, v, device))
        if metrics['accuracy'][-1] >= best_val_acc:
            best_val_acc = metrics['accuracy'][-1]
            counter = 0
            print(f"Epoch {epoch+1}: New best accuracy: {best_val_acc:.4f} saving model...")
            state = {
                'epoch': epoch,
                'state_dict': model.state_dict(),
                'optimizer': optimizer.state_dict()
            }
            state.update({k: metrics[k][-1] for k in metrics.keys()})
            torch.save(state, path)
        else:
            counter += 1
        if counter >= patience:
            print(f"Epoch {epoch+1}: Early stop triggered.")
            break
    return metrics

In [ ]:
from models.resnet import resnet34, resnet50
from utils.plots import plot_training_metrics
from utils.losses import DistillationLoss
import torch.optim as optim

collection = []
As = [0.2, 0.8]
for a in As:
    model = resnet50().to(device)
    criterion = DistillationLoss(T=3.0, alpha=a)
    optimizer = optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)
    path = f'../models/weights/cifar10_resnet50_a{int(a*100)}.pth'
    metrics = distill_val(train_loader, val_loader, model, criterion, optimizer, scheduler, 
                        device=device, path=path, patience=50, epochs=50)
    collection.append(metrics)
    with open(os.path.join(project_dir, f'data/states/cifar10_resnet50_a{int(a*100)}.json'), 'w') as f:
        json.dump(metrics, f) 

training...:  79%|███████▉  | 309/391 [01:07<00:16,  4.85it/s]

In [ ]:
import matplotlib.pyplot as plt

# modified
num_metrics = len(metrics)
fig, axes = plt.subplots(num_metrics, 1, figsize=(8, 4 * num_metrics))
if num_metrics == 1:
    axes = [axes]
epochs = range(1, len(next(iter(metrics.values()))) + 1)
for i, k in enumerate(metrics.keys()):
    for j in range(len(collection)):
        axes[i].plot(epochs, collection[j][k], label=f"alpha = {As[j]}")
    axes[i].set_title(k)
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel(k)
    axes[i].legend()
    axes[i].grid(True)
plt.tight_layout()
plt.show()

# Long

In [ ]:
from models.resnet import resnet34, resnet50
from utils.plots import plot_training_metrics
from utils.losses import DistillationLoss
import torch.optim as optim

a = 0.5
model = resnet50().to(device)
criterion = DistillationLoss(T=3.0, alpha=a)
optimizer = optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)
path = f'../models/weights/cifar10_resnet50_a{int(a*100)}_long.pth'
metrics = distill_val(train_loader, val_loader, model, criterion, optimizer, scheduler, 
                    device=device, path=path, patience=5, epochs=100)
with open(os.path.join(project_dir, f'data/states/cifar10_resnet50_a{int(a*100)}_long.json'), 'w') as f:
    json.dump(metrics, f) 
plot_training_metrics(metrics)

# Evaluations

In [ ]:
a30 = torch.load(f'../models/weights/cifar10_resnet50_a30.pth', map_location=device)
print(f"Transfer: Val Accuracy: {a30['accuracy']:.4f}")

model.load_state_dict(a30['state_dict'])
a30_acc = evaluate_model(test_loader, model, Accuracy(), device)
print(f"Alpha = 0.30: Test Accuracy: {a30_acc:.4f}")

/tmp/ipykernel_765278/782345861.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  a30 = torch.load(f'../models/weights/cifar10_resnet50_a30.pth', map_location=device)


Transfer: Val Accuracy: 0.5913
evaluating...: 100%|██████████| 63/63 [00:04<00:00, 12.93it/s]
Transfer: Test Accuracy: 0.5856


In [ ]:
a70 = torch.load(f'../models/weights/cifar10_resnet50_a70.pth', map_location=device)
print(f"Transfer: Val Accuracy: {a70['accuracy']:.4f}")

model.load_state_dict(a70['state_dict'])
a70_acc = evaluate_model(test_loader, model, Accuracy(), device)
print(f"Alpha = 0.70: Test Accuracy: {a70_acc:.4f}")